In [2]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

roll_number = 58
digits = [int(d) for d in str(roll_number)[-2:]]

categories = ["billing", "account", "general"]

personalized_entries = []
for d in digits:
    cat = categories[d % 3]
    if cat == "billing":
        personalized_entries.append({
            "question": "how do I download my fee receipt",
            "answer": "You can download it from the student portal under Billing section.",
            "keywords": "receipt fee billing download",
            "category": "billing"
        })
    elif cat == "account":
        personalized_entries.append({
            "question": "how do I update my registered mobile number",
            "answer": "Go to Account Settings > Update Mobile.",
            "keywords": "account mobile update change",
            "category": "account"
        })
    else:  # general
        personalized_entries.append({
            "question": "where can I find the academic calendar",
            "answer": "The academic calendar is available on the college website.",
            "keywords": "calendar academic schedule general",
            "category": "general"
        })

faq_entries = fixed_entries + personalized_entries
df = pd.DataFrame(faq_entries)
df


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where can I find the academic calendar,The academic calendar is available on the coll...,calendar academic schedule general,general
5,where can I find the academic calendar,The academic calendar is available on the coll...,calendar academic schedule general,general


In [3]:
def score_query(query, df):
    results = []
    for idx, row in df.iterrows():
        score = sum([1 for word in query.lower().split() if word in row["keywords"].lower().split()])
        if score > 0:
            results.append((row["question"], row["answer"], score))
    return sorted(results, key=lambda x: x[2], reverse=True)

score_query("fee payment", df)


[('how can i pay the fee', 'You can pay via UPI, card, or net banking.', 2),
 ('what is the annual fee', 'The annual fee is Rs 500.', 1)]

In [4]:
def same_category(category_name, df):
    return df[df["category"] == category_name]
    
same_category("billing", df)


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing


In [5]:
new_keyword = "charges"
df.at[0, "keywords"] += " " + new_keyword

df.to_csv("23_faq_data.csv", index=False)  


In [6]:
df.groupby("category").size()

category
account    1
billing    2
general    3
dtype: int64

In [7]:
def score_query_with_ties(query, df):
    results = []
    for idx, row in df.iterrows():
        score = sum([1 for word in query.lower().split() if word in row["keywords"].lower().split()])
        if score > 0:
            results.append((row["question"], row["answer"], score))
    if not results:
        return []
    results.sort(key=lambda x: x[2], reverse=True)
    max_score = results[0][2]
    return [r for r in results if r[2] == max_score]

print("Tie case:")
print(score_query_with_ties("fee", df))  

print("\nNon-tie case:")
print(score_query_with_ties("password", df)) 


Tie case:
[('what is the annual fee', 'The annual fee is Rs 500.', 1), ('how can i pay the fee', 'You can pay via UPI, card, or net banking.', 1)]

Non-tie case:
[('how to reset password', 'Go to Settings > Reset Password.', 1)]
